# Лабораторная работа № 5

## Криптоанализ шифра RSA

**Цель работы:** исследование стойкости алгоритма RSA к атаке на факторизацию модуля.

Программа позволяет:

1. проверять работу расширенного алгоритма Евклида и модульного возведения в степень;
2. проверять факторизацию числа `n` методом группового пробного деления;
3. генерировать пары ключей RSA;
4. шифровать и расшифровывать сообщения;
5. по известному открытому ключу `(e, n)` восстанавливать закрытый ключ
   `d = e^(-1) mod φ(n)` и расшифровывать перехваченное сообщение.

Все НОД в программе вычисляются через собственный `extended_gcd`.


In [16]:
import math
import random
import json
import time

In [17]:
def extended_gcd(a, b):
    sa = 1 if a >= 0 else -1
    sb = 1 if b >= 0 else -1
    a, b = abs(a), abs(b)

    if a == 0:
        return b, 0, sb

    g, x1, y1 = extended_gcd(b % a, a)
    x = y1 - (b // a) * x1
    y = x1

    return g, x * sa, y * sb


def gcd(a, b):
    """НОД через расширенный алгоритм Евклида (без math.gcd)."""
    return extended_gcd(a, b)[0]


def mod_inverse(a, m):
    a %= m
    g, x, _ = extended_gcd(a, m)

    return x % m if g == 1 else None


def pow_mod(base, exp, mod):
    result = 1
    base %= mod
    while exp > 0:
        if exp % 2 == 1:
            result = (result * base) % mod
        exp //= 2
        base = (base * base) % mod
    return result

In [18]:
PRIMES_CACHE = {}


def generate_primes(limit):
    """Генерирует простые числа от 2 до limit решетом Эратосфена."""
    if limit < 2:
        return []

    sieve = bytearray([1]) * (limit + 1)
    sieve[0:2] = b"\x00\x00"
    for i in range(2, math.isqrt(limit) + 1):
        if sieve[i]:
            sieve[i * i::i] = bytearray(len(sieve[i * i::i]))

    return [i for i in range(2, limit + 1) if sieve[i]]


def get_primes_up_to(limit):
    """Возвращает простые числа до limit с кешированием между вызовами."""
    if limit not in PRIMES_CACHE:
        PRIMES_CACHE[limit] = generate_primes(limit)
    return PRIMES_CACHE[limit]


def is_prime(n):
    if n < 2:
        return False
    for i in range(2, math.isqrt(n) + 1):
        if n % i == 0:
            return False
    return True


def split_primes_into_groups(primes, batch_size):
    """Группирует простые числа: [2, 3, 5], [7, 11, 13], ..."""
    return [primes[i:i + batch_size] for i in range(0, len(primes), batch_size)]


def find_divisor_in_group(n, group):
    """Ищет делитель n внутри подозрительной группы.

    Сначала группа делится пополам (снова через НОД), затем проверяются
    отдельные числа. Возвращает найденный простой делитель или None.
    """
    if not group:
        return None

    if len(group) == 1:
        return group[0] if n % group[0] == 0 else None

    middle = len(group) // 2
    for part in (group[:middle], group[middle:]):
        product = 1
        for prime in part:
            product *= prime
        if gcd(n, product) != 1:
            found = find_divisor_in_group(n, part)
            if found is not None:
                return found

    for prime in group:
        if n % prime == 0:
            return prime

    return None


def factorize_by_trial_division(n, batch_size=3, verbose=True):
    """Полная факторизация n пробными делениями"""
    if n < 2:
        raise ValueError("n должно быть целым числом больше 1")

    start = time.perf_counter()
    current = n
    factors = []

    limit = math.isqrt(n)
    primes = get_primes_up_to(limit)
    groups = split_primes_into_groups(primes, batch_size)

    if verbose:
        print(f"n = {n}, sqrt(n) = {math.sqrt(n):.3f}, "
              f"простых до sqrt(n): {len(primes)}")
        print(f"Группировка по {batch_size}: {len(groups)} групп")

    for group in groups:
        product = 1
        for prime in group:
            product *= prime

        common = gcd(current, product)

        if verbose:
            group_str = " * ".join(str(x) for x in group)
            print(f"  Q = {group_str} = {product}; НОД({current}, Q) = {common}"
                  + ("" if common == 1 else "  <-- делитель в группе"))

        while common != 1:
            divisor = find_divisor_in_group(current, group)
            if divisor is None:
                break

            factors.append(divisor)
            current //= divisor

            if verbose:
                print(f"    Найден делитель: {divisor}, новый остаток: {current}")

            if current == 1:
                break

            common = gcd(current, product)

        if current == 1:
            break

    if current > 1:
        factors.append(current)

    factors.sort()
    elapsed = time.perf_counter() - start

    if not all(is_prime(factor) for factor in factors):
        raise ValueError(f"n = {n}: среди множителей {factors} есть составные")

    if math.prod(factors) != n:
        raise ValueError(f"n = {n}: произведение множителей {factors} не равно n")

    if verbose:
        print(f"\nРазложение: {n} = {' * '.join(str(f) for f in factors)}")
        print(f"Время факторизации: {elapsed:.6f} с")

    return {
        'n': n,
        'factors': factors,
        'elapsed': elapsed,
    }

In [ ]:
def factorization_check_menu():
    """Проверка факторизации без ключа и без вычисления d."""
    print("\n--- ПРОВЕРКА ФАКТОРИЗАЦИИ ---")
    try:
        n = int(input("Введите целое число n: "))
    except ValueError:
        print("Ошибка ввода: нужно целое число.")
        return

    size_input = input("Размер группы [по умолч.: 3]: ").strip()
    try:
        batch_size = int(size_input) if size_input else 3
    except ValueError:
        print("Ошибка ввода размера группы.")
        return

    if batch_size < 1:
        print("Ошибка: размер группы должен быть положительным.")
        return

    try:
        result = factorize_by_trial_division(n, batch_size=batch_size)
    except ValueError as ve:
        print(f"Ошибка факторизации: {ve}")
        return

    factors = result['factors']
    factors_str = ' * '.join(str(factor) for factor in factors)

    print(f"\nРезультат: n = {factors_str}")

In [20]:
def check_math_menu():
    print("\n=== ПРОВЕРКА МАТЕМАТИКИ ===")
    choice = input("1. Алгоритм Евклида  2. Возведение в степень  "
                   "3. Факторизация  0. Назад\nВыбор: ").strip()

    if choice == '1':
        try:
            a = int(input("Введите число a: "))
            b = int(input("Введите число b: "))

            g, x, y = extended_gcd(a, b)

            print(f"Вход: a={a}, b={b}")
            print(f"НОД({a}, {b}) = {g}")
            print(f"Коэффициенты Безу: x={x}, y={y}")
            print(f"Проверка тождества Безу: {a}*({x}) + {b}*({y}) = {a * x + b * y}")

        except ValueError:
            print("Ошибка ввода: необходимо ввести целые числа.")

    elif choice == '2':
        try:
            base = int(input("Введите основание (base): "))
            exp = int(input("Введите показатель степени (exp): "))
            mod = int(input("Введите модуль (mod): "))

            if mod <= 0:
                print("Ошибка: модуль должен быть положительным числом.")
                return

            result = pow_mod(base, exp, mod)

            print(f"{base}^{exp} mod {mod} = {result}")

        except ValueError:
            print("Ошибка ввода: необходимо ввести целые числа.")

    elif choice == '3':
        factorization_check_menu()

    elif choice == '0':
        print("Возврат в главное меню.")
        return False

    else:
        print("Неверный выбор. Попробуйте снова.")

    return True

In [21]:
def is_prime(n):
    if n < 2:
        return False
    for i in range(2, math.isqrt(n) + 1):
        if n % i == 0:
            return False
    return True


def generate_random_e(phi_n):
    while True:
        e = random.randrange(3, phi_n, 2)
        if gcd(e, phi_n) == 1:
            return e


def generate_rsa_keypair(p, q, e):
    if not (is_prime(p) and is_prime(q)):
        raise ValueError("p и q должны быть простыми")

    n = p * q
    phi_n = (p - 1) * (q - 1)

    if gcd(e, phi_n) != 1:
        raise ValueError(f"e={e} не взаимно просто с φ(n)")

    d = mod_inverse(e, phi_n)
    if d is None:
        raise ValueError("Не удалось найти обратный элемент d")

    return {
        'n': n,
        'e': e,
        'd': d
    }

In [22]:
def generate_keys_menu():
    print("\n=== ГЕНЕРАЦИЯ КЛЮЧЕЙ RSA ===")
    
    p_input = input("Введите простое число p [по умолч.: 227]: ").strip()
    p = int(p_input) if p_input else 227
    
    q_input = input("Введите простое число q [по умолч.: 373]: ").strip()
    q = int(q_input) if q_input else 373
    
    keys = []
    try:
        for i in range(1, 4):
            print(f"\n--- Генерация пары ключей #{i} ---")
            
            e_str = input(f"Введите открытую экспоненту e для пары #{i} [Enter = случайно]: ").strip()
            
            if not e_str:
                phi_n_temp = (p - 1) * (q - 1)
                e = generate_random_e(phi_n_temp)
                print(f"Случайно сгенерировано e = {e}")
            else:
                e = int(e_str)
            
            kp = generate_rsa_keypair(p, q, e)
            keys.append(kp)
            
            print(f"Пара #{i}: e={kp['e']}, d={kp['d']}, n={kp['n']}")
            
        filename = "rsa_keys.json"
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump({
                'params': {'p': p, 'q': q},
                'keys': keys
            }, f, indent=2, ensure_ascii=False)
                
        print(f"\nКлючи сохранены в '{filename}'")
        
    except ValueError as ve:
        print(f"Ошибка генерации: {ve}")
    except Exception as ex:
        print(f"Непредвиденная ошибка: {ex}")

In [23]:
ALPHABET_RU = "АБВГДЕЖЗИЙКЛМНОПРСТУФХЦЧШЩЪЫЬЭЮЯ"
CHAR_TO_CODE = {char: str(10 + i) for i, char in enumerate(ALPHABET_RU)}
CHAR_TO_CODE[' '] = '99'

def prepare_text(text):
    """Подготовка открытого текста: верхний регистр, только русские буквы и пробелы"""
    cleaned = ""
    for char in text.upper():
        if char in ALPHABET_RU or char == ' ':
            cleaned += char
    return cleaned

def text_to_numeric_string(text):
    """Преобразование текста в строку цифровых кодов"""
    numeric_parts = []
    for char in text:
        code = CHAR_TO_CODE.get(char)
        if code is not None:
            numeric_parts.append(code)
    return "".join(numeric_parts)

In [24]:
def split_into_blocks(numeric_str, n):
    """Разбивает числовую строку на блоки, каждый из которых меньше n.

    Блок не может начинаться с нуля: иначе при обратном склеивании
    (int -> str) ведущий ноль потеряется и текст не восстановится.
    """
    if n <= 1:
        raise ValueError("Модуль n должен быть больше 1")

    blocks = []
    max_len = len(str(n))
    i = 0
    while i < len(numeric_str):
        block_len = min(max_len, len(numeric_str) - i)
        while block_len > 1:
            candidate = numeric_str[i:i + block_len]
            next_idx = i + block_len
            if int(candidate) < n and (next_idx >= len(numeric_str) or numeric_str[next_idx] != '0'):
                break
            block_len -= 1
        blocks.append(int(numeric_str[i:i + block_len]))
        i += block_len

    if "".join(str(b) for b in blocks) != numeric_str:
        raise ValueError(
            "Модуль n слишком мал для двузначных кодов: "
            "разбиение теряет ведущие нули"
        )

    return blocks

In [25]:
def encrypt_menu():
    print("\n=== ШИФРОВАНИЕ RSA ===")
    try:
        with open("rsa_keys.json", encoding="utf-8") as f:
            keys = json.load(f)["keys"]
    except (FileNotFoundError, KeyError, ValueError):
        keys = []

    e = n = None
    if keys:
        key_list = "\n".join(
            f"{i}. e={k['e']}, n={k['n']}"
            for i, k in enumerate(keys, 1)
        )
        choice = input(f"Доступные ключи:\n{key_list}\nНомер ключа [Enter = ввести свой]: ").strip()
        if choice:
            try:
                idx = int(choice) - 1
                if 0 <= idx < len(keys):
                    e, n = keys[idx]['e'], keys[idx]['n']
                else:
                    print("Неверный номер ключа.")
                    return
            except ValueError:
                print("Ошибка ввода номера.")
                return
    else:
        print("Файл ключей не найден — введите ключ вручную.")

    if e is None:
        try:
            e = int(input("Введите e: "))
            n = int(input("Введите n: "))
        except ValueError:
            print("Ошибка ввода: нужны целые числа.")
            return

    print(f"Ключ: e={e}, n={n}")
    text = prepare_text(input("Открытый текст: "))
    if not text:
        print("Текст пуст.")
        return

    try:
        blocks = split_into_blocks(text_to_numeric_string(text), n)
    except ValueError as ve:
        print(f"Ошибка: {ve}")
        return

    print(f"Блоки открытого текста: {format_blocks(blocks)}")

    block_len = len(str(n))
    cipher = [pow_mod(m, e, n) for m in blocks]
    print(f"ШИФР-ТЕКСТ: {format_blocks(cipher, block_len)}")

In [26]:
CODE_TO_CHAR = {v: k for k, v in CHAR_TO_CODE.items()}
VALID_CODES = set(CHAR_TO_CODE.values())


def format_blocks(values, width=0):
    """Представляет блоки строкой, разделённой символом |."""
    if width:
        return " | ".join(str(value).zfill(width) for value in values)
    return " | ".join(str(value) for value in values)


def parse_cipher_blocks(cipher_str, n):
    """Разбирает шифр-текст на числовые блоки.

    Поддерживаются два формата:
    - с разделителями: "25160 | 33284";
    - без разделителей: "2516033284" — по len(str(n)) цифр на блок.
    Возвращает список чисел или None, если формат некорректен.
    """
    if n <= 1:
        return None

    if "|" in cipher_str:
        chunks = [chunk.strip() for chunk in cipher_str.split("|")]
        if any(not chunk.isdigit() for chunk in chunks):
            return None
    else:
        compact = cipher_str.replace(" ", "")
        if not compact.isdigit():
            return None
        width = len(str(n))
        if len(compact) % width != 0:
            return None
        chunks = [compact[i:i + width] for i in range(0, len(compact), width)]

    if not chunks:
        return None

    values = [int(chunk) for chunk in chunks]
    if any(value >= n for value in values):
        return None

    return values


def find_cipher_blocks(cipher_str, d, n):
    """Расшифровывает шифр-текст по блокам, найденным в его записи."""
    values = parse_cipher_blocks(cipher_str, n)
    if values is None:
        return None

    decrypted = [pow_mod(value, d, n) for value in values]
    digits = "".join(str(block) for block in decrypted)
    if len(digits) % 2 != 0:
        return None
    if any(digits[i:i + 2] not in VALID_CODES for i in range(0, len(digits), 2)):
        return None

    return decrypted


def decrypt_blocks_to_text(decrypted_blocks):
    """Склеивает блоки и разбирает цифровую строку по парам кодов."""
    numeric_str = "".join(str(b) for b in decrypted_blocks)
    text = ""
    for i in range(0, len(numeric_str) - 1, 2):
        text += CODE_TO_CHAR.get(numeric_str[i:i + 2], '?')
    return text

In [27]:
def decrypt_menu():
    print("\n=== РАСШИФРОВАНИЕ RSA ===")
    try:
        with open("rsa_keys.json", encoding="utf-8") as f:
            keys = json.load(f)["keys"]
    except (FileNotFoundError, KeyError, ValueError):
        keys = []

    d = n = None
    if keys:
        key_list = "\n".join(
            f"{i}. d={k['d']}, n={k['n']}"
            for i, k in enumerate(keys, 1)
        )
        choice = input(f"Доступные ключи:\n{key_list}\nНомер ключа [Enter = ввести свой]: ").strip()
        if choice:
            try:
                idx = int(choice) - 1
                if 0 <= idx < len(keys):
                    d, n = keys[idx]['d'], keys[idx]['n']
                else:
                    print("Неверный номер ключа.")
                    return
            except ValueError:
                print("Ошибка ввода номера.")
                return
    else:
        print("Файл ключей не найден — введите ключ вручную.")

    if d is None:
        try:
            d = int(input("Введите d: "))
            n = int(input("Введите n: "))
        except ValueError:
            print("Ошибка ввода: нужны целые числа.")
            return

    print(f"Ключ: d={d}, n={n}")
    s = input("Шифр-текст (блоки через | или одной строкой цифр): ").strip()
    if not s:
        print("Шифр-текст пуст.")
        return
    if any(ch not in "0123456789| \t" for ch in s):
        print("Ошибка: шифр-текст должен состоять из цифр и разделителей |.")
        return

    blocks = find_cipher_blocks(s, d, n)
    if blocks is None:
        print("Не удалось разобрать шифр-текст на блоки: проверьте разделители и модуль n.")
        return

    print(f"Расшифрованные блоки: {blocks}")
    text = decrypt_blocks_to_text(blocks)
    print(f"ОТКРЫТЫЙ ТЕКСТ: {text}")


## Криптоанализ RSA методом группового пробного деления

Факторизация находит полное разложение числа на простые множители с учётом
кратностей, например `98 = 2 * 7 * 7` или `45 = 3 * 3 * 5`. На каждом шаге
для текущего остатка ищется один простой делитель, остаток делится на него,
и поиск повторяется, пока остаток не станет равным 1. Пригодность разложения
для RSA проверяется отдельно, в `recover_private_key`.

Модуль RSA имеет вид `n = p * q` — ровно два простых множителя. Для такого
модуля по открытому ключу `(e, n)` можно вычислить:

* `φ(n) = (p - 1)(q - 1)` (в общем случае `φ(n) = Π p^(k-1) (p - 1)`);
* `d = e^(-1) mod φ(n)`.

После этого перехваченный шифр-текст расшифровывается так же, как при наличии
легального закрытого ключа. Если факторизация дала не два множителя
(например, `98 = 2 * 7 * 7`), RSA-ключ не восстанавливается — само разложение
при этом найдено корректно.

Простые числа генерируются решетом Эратосфена до `√остатка` и группируются
по 3:

```
(2, 3, 5) -> Q1 = 30
(7, 11, 13) -> Q2 = 1001
(17, 19, 23) -> Q3 = 7429
```

Для каждой группы достаточно одной проверки `НОД(остаток, Qi)`:

* если `НОД(остаток, Qi) = 1`, в группе точно нет делителя остатка — вся группа
  отбрасывается;
* если `НОД(остаток, Qi) > 1`, группа подозрительная и уточняется делением
  пополам.

Такой приём сокращает число операций по сравнению с проверкой каждого числа
подряд. Все НОД вычисляются своим `extended_gcd`, без `math.gcd`.

In [28]:
RSA_FACTORS_COUNT = 2


def is_rsa_factorization(factors):
    """Модель RSA: ровно два простых множителя (с учётом кратностей)."""
    return len(factors) == RSA_FACTORS_COUNT


def phi_from_factors(factors):
    """φ(n) = Π p^(k-1) * (p - 1) для n = Π p^k."""
    phi = 1
    for prime in set(factors):
        power = factors.count(prime)
        phi *= prime ** (power - 1) * (prime - 1)
    return phi


def recover_private_key(e, n, batch_size=3):
    """Восстанавливает закрытый ключ по открытому ключу (e, n)."""
    if e <= 1:
        raise ValueError("e должно быть больше 1")

    result = factorize_by_trial_division(n, batch_size=batch_size)
    factors = result['factors']
    factors_str = ' * '.join(str(factor) for factor in factors)

    if not is_rsa_factorization(factors):
        raise ValueError(
            f"n = {n} разложено как {factors_str}, но для RSA нужен "
            f"модуль ровно из двух простых множителей; "
            f"найдено множителей: {len(factors)}"
        )

    p, q = factors
    phi_n = phi_from_factors(factors)

    if p == q:
        print(f"φ(n) = φ({p}^2) = {p} * {p - 1} = {phi_n}")
    else:
        print(f"φ(n) = (p - 1)(q - 1) = {p - 1} * {q - 1} = {phi_n}")

    d = mod_inverse(e, phi_n)
    if d is None:
        raise ValueError(f"e = {e} не взаимно просто с φ(n) = {phi_n}")

    print(f"Закрытая экспонента: d = e^(-1) mod φ(n) = {d}")

    return {
        'n': n,
        'e': e,
        'd': d,
        'p': p,
        'q': q,
        'phi': phi_n,
        'factors': factors,
    }

In [29]:
def cryptoanalysis_menu():
    print("\n=== КРИПТОАНАЛИЗ RSA: ФАКТОРИЗАЦИЯ МОДУЛЯ n ===")

    keys = []
    try:
        with open("rsa_keys.json", encoding="utf-8") as f:
            keys = json.load(f)["keys"]
    except (FileNotFoundError, KeyError, ValueError):
        keys = []

    e = n = None
    if keys:
        unique_keys = []
        for k in keys:
            pair = (k['e'], k['n'])
            if pair not in unique_keys:
                unique_keys.append(pair)

        key_list = "\n".join(
            f"{i}. e={key_e}, n={key_n}"
            for i, (key_e, key_n) in enumerate(unique_keys, 1)
        )
        choice = input(
            f"Известные открытые ключи:\n{key_list}\n"
            "Номер ключа [Enter = ввести вручную]: "
        ).strip()

        if choice:
            try:
                idx = int(choice) - 1
                if 0 <= idx < len(unique_keys):
                    e, n = unique_keys[idx]
                else:
                    print("Неверный номер ключа.")
                    return
            except ValueError:
                print("Ошибка ввода номера.")
                return
    else:
        print("Файл rsa_keys.json не найден — введите открытый ключ вручную.")

    if n is None:
        try:
            e = int(input("Введите открытую экспоненту e: "))
            n = int(input("Введите модуль n: "))
        except ValueError:
            print("Ошибка ввода: нужны целые числа.")
            return

    print(f"Открытый ключ: e={e}, n={n}")

    size_input = input("Размер группы для факторизации [по умолч.: 3]: ").strip()
    try:
        batch_size = int(size_input) if size_input else 3
    except ValueError:
        print("Ошибка ввода размера группы.")
        return
    if batch_size < 1:
        print("Ошибка: размер группы должен быть положительным.")
        return

    s = input("Шифр-текст (блоки через | или одной строкой цифр): ").strip()
    if not s:
        print("Шифр-текст пуст.")
        return
    if any(ch not in "0123456789| \t" for ch in s):
        print("Ошибка: шифр-текст должен состоять из цифр и разделителей |.")
        return

    try:
        recovered = recover_private_key(e, n, batch_size=batch_size)
    except ValueError as ve:
        print(f"Атака не удалась: {ve}")
        return

    print(f"\nВосстановленный закрытый ключ: d={recovered['d']}, n={recovered['n']}")

    blocks = find_cipher_blocks(s, recovered['d'], recovered['n'])
    if blocks is None:
        print("Не удалось разобрать шифр-текст на блоки: проверьте разделители и модуль n.")
        return

    print(f"Расшифрованные блоки: {blocks}")
    print(f"ОТКРЫТЫЙ ТЕКСТ: {decrypt_blocks_to_text(blocks)}")

In [30]:
def crypto_menu():
    while True:
        print("\n=== ШИФРОВАНИЕ / РАСШИФРОВАНИЕ ===")
        choice = input("1. Зашифровать  2. Расшифровать  0. Назад\nВыбор: ").strip()

        if choice == '1':
            encrypt_menu()
        elif choice == '2':
            decrypt_menu()
        elif choice == '0':
            break
        else:
            print("Неверный выбор, попробуйте снова.")


def main_menu():
    while True:
        print("\n=== ГЛАВНОЕ МЕНЮ (RSA) ===")
        choice = input("1. Проверка математики  2. Генерация ключей  "
                       "3. Шифрование/расшифрование  "
                       "4. Криптоанализ (факторизация n)  0. Выход\nВыбор: ").strip()

        if choice == '1':
            while check_math_menu():
                pass
        elif choice == '2':
            generate_keys_menu()
        elif choice == '3':
            crypto_menu()
        elif choice == '4':
            cryptoanalysis_menu()
        elif choice == '0':
            print("Выход из программы.")
            break
        else:
            print("Неверный выбор, попробуйте снова.")


main_menu()



=== ГЛАВНОЕ МЕНЮ (RSA) ===

=== ПРОВЕРКА МАТЕМАТИКИ ===

--- ПРОВЕРКА ФАКТОРИЗАЦИИ ---
Ошибка: размер группы должен быть положительным.

=== ПРОВЕРКА МАТЕМАТИКИ ===
Ошибка ввода: необходимо ввести целые числа.

=== ПРОВЕРКА МАТЕМАТИКИ ===
Неверный выбор. Попробуйте снова.

=== ПРОВЕРКА МАТЕМАТИКИ ===

--- ПРОВЕРКА ФАКТОРИЗАЦИИ ---
n = 121, sqrt(n) = 11.000, простых до sqrt(n): 5
Группировка по 3: 2 групп
  Q = 2 * 3 * 5 = 30; НОД(121, Q) = 1
  Q = 7 * 11 = 77; НОД(121, Q) = 11  <-- делитель в группе
    Найден делитель: 11, новый остаток: 11
    Найден делитель: 11, новый остаток: 1

Разложение: 121 = 11 * 11
Время факторизации: 0.001022 с

Результат: n = 11 * 11
Проверка: 11 * 11 = 121
Простых множителей (с кратностями): 2
Ровно два простых множителя (как у модуля RSA): да

=== ПРОВЕРКА МАТЕМАТИКИ ===

--- ПРОВЕРКА ФАКТОРИЗАЦИИ ---
n = 576, sqrt(n) = 24.000, простых до sqrt(n): 9
Группировка по 3: 3 групп
  Q = 2 * 3 * 5 = 30; НОД(576, Q) = 6  <-- делитель в группе
    Найден делител